# 04 - TF-IDF counterfactuals (GPU)

**Settings:** Accelerator **GPU T4 x1** (x2 also fine) | Internet **On** | Secret `HF_TOKEN` on (optional: the ungated `unsloth` mirror of Llama-3.2-3B is used otherwise). No dataset input needed: the code is cloned from GitHub.

**Runtime:** ~3 min candidates (CPU) + ~10 min preflight + ~1.5-2.5 h main run. Every step resumes if re-run.

**What it makes:** minimal-pair counterfactuals in the **TF-IDF** setting, for causal patching (RQ2) and as a separate evaluation stratum for the probe (they are **not** used to train the probe).
* Seed = a TF-IDF question whose hop 1 was correct. Its entry entity ("Heads Or Tails (1937 Film)") is swapped for the entity of another question with the **same template** (same entity type), **same split**, **same token length**, whose evidence is **not** in the seed's context.
* The CF prompt is the seed's stored prompt with only those entity tokens changed (same system prompt, date token and TF-IDF context), so it is an exact token-aligned minimal pair.
* Generation, hop labeling (against the donor question's real gold chain) and hidden-state extraction use the **Phase 1 code itself**. Only induced hop-1 failures are kept; a CF where the model still names the donor's correct entity (memory) is rejected.

**Check in the preflight output (STEP 2) before the main run continues:**
* `HIDDEN-STATE PARITY ... [0.9999.., ...]` - must be >= 0.999 (it aborts otherwise)
* `TOKEN ALIGNMENT: 0 misaligned`
* `GENERATION REPRODUCTION: k/4` - informational (most should be identical)
* whether Wikidata works (a WARNING box means labels are made without aliases, as in the Phase 1 TF-IDF run)

Tip: after the preflight passed once, use **Save Version -> Save & Run All (Commit)** so the main run goes in the background. Download `cf_tfidf_results.zip` from the Output tab.

In [ ]:
# ===== Internet check (Settings -> Internet must be On; committed runs use the SAVED setting) =====
import socket
for host in ('github.com', 'huggingface.co', 'query.wikidata.org'):
    try:
        socket.gethostbyname(host)
    except OSError:
        raise SystemExit(f'No internet ({host} does not resolve). Turn Internet ON in the notebook settings, then run again.')
print('internet OK')

In [ ]:
# ===== Setup: clone the code, HF token, GPU =====
!pip install -q sentence-transformers
import os, subprocess, sys, json
REPO_URL = 'https://github.com/ANLPproject/LinguaFranca.git'
BRANCH = 'cf-tfidf'
WORK = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull'], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'No HF_TOKEN secret ({e.__class__.__name__}); the ungated model mirror will be used')
import torch
assert torch.cuda.is_available(), 'No GPU: Session options -> Accelerator -> GPU T4 x1, then restart.'
print('CUDA:', torch.cuda.device_count(), 'x', torch.cuda.get_device_name(0))

def run(cmd):
    """Stream a command's output; return its exit code."""
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        if 'it/s]' in line or 's/it]' in line:
            continue
        print(line, end='')
    return p.wait()

OUT = '/kaggle/working/cf_tfidf'
PRE = '/kaggle/working/cf_preflight'

In [ ]:
# ===== STEP 1 - candidates (CPU, ~3 min): seeds, same-template donors, token-alignment check =====
CAND = 'data/cf_tfidf/candidates.jsonl'
if not os.path.exists(CAND):
    rc = run([sys.executable, 'scripts/make_cf_candidates.py', '--skip-aliases', '--out', 'data/cf_tfidf'])
    assert rc == 0, 'candidate building failed - see the log above'
st = json.load(open('data/cf_tfidf/candidates.stats.json'))
print('\nseeds with >=1 valid donor:', st['n_seeds_with_donor'], st['by_split'])
print('rejected donors:', st['rejected_donors'])
assert st['n_seeds_with_donor'] >= 600, 'too few candidate seeds - something is off, do not continue'

In [ ]:
# ===== STEP 2 - PREFLIGHT (~10 min): self-checks + 4 accepted CFs. Aborts loudly if anything is wrong. =====
rc = run([sys.executable, 'generate_counterfactuals_tfidf.py', '--candidates', CAND, '--out', PRE,
          '--target', '4', '--batch-size', '4', '--n-repro', '4'])
assert rc == 0 and os.path.exists(f'{PRE}/summary.json'), 'PREFLIGHT FAILED - read the log above. Nothing long was started.'
print('\nPREFLIGHT PASSED')

In [ ]:
# ===== STEP 3 - MAIN RUN (~1.5-2.5 h on a T4; resumable: re-running continues where it stopped) =====
TARGET = 500
for attempt in range(1, 4):
    rc = run([sys.executable, 'generate_counterfactuals_tfidf.py', '--candidates', CAND, '--out', OUT,
              '--target', str(TARGET), '--batch-size', '8', '--max-minutes', '600'])
    if rc == 0:
        break
    print(f'\n[attempt {attempt}] exit code {rc} - resuming (finished seeds are skipped)')
print('main run exit code:', rc)

In [ ]:
# ===== STEP 4 - package (always run this) =====
import zipfile, shutil
shutil.copy('data/cf_tfidf/candidates.stats.json', OUT)
zpath = '/kaggle/working/cf_tfidf_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, os.path.dirname(OUT)))
print(f'{zpath}: {os.path.getsize(zpath) / 1e6:.1f} MB')
print(open(f'{OUT}/summary.md', encoding='utf-8').read())

In [ ]:
# ===== STEP 5 (optional) - publish to the team HF dataset under a NEW folder (never overwrites existing files) =====
# Needs HF_TOKEN with WRITE access to AnishRacherla/LinguaFranca-Phase3. Leave False to just download the zip.
UPLOAD = False
if UPLOAD:
    from huggingface_hub import HfApi
    HfApi().upload_folder(repo_id='AnishRacherla/LinguaFranca-Phase3', repo_type='dataset', folder_path=OUT,
                          path_in_repo='tfidf_distractor_rag/cf_v1',
                          allow_patterns=['counterfactuals.jsonl', 'hidden_states/*.pt', 'attempts.jsonl',
                                          'summary.json', 'summary.md', 'candidates.stats.json', 'aliases.json'],
                          commit_message='Add TF-IDF counterfactuals cf_v1 (new folder; no existing files changed)')
    print('uploaded to tfidf_distractor_rag/cf_v1')